# Marketplace Safety – Machine Learning

## Gruppuppgift / individuell inlämning

I det här projektet bygger jag en lösning för att hjälpa ett Trust & Safety team att prioritera misstänkta händelser i en marketplace-app.

Varje rad i datasetet representerar en händelse på plattformen, till exempel en annons eller ett meddelande. Målet är att klassificera om händelsen är misstänkt eller inte.

Target-variabeln är:

- `is_suspicious = 1`: misstänkt händelse
- `is_suspicious = 0`: ej misstänkt händelse

Eftersom uppgiften handlar om prioritering är målet inte att få 100% rätt, utan att bygga en modell som fungerar rimligt bra, går att förklara och kan användas som beslutsstöd.

In [1]:
# Grundläggande bibliotek
from pathlib import Path

import numpy as np
import pandas as pd

# Visualisering
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# Inställningar
RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

Matplotlib is building the font cache; this may take a moment.


In [2]:
# Sökvägar
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "historical_data.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH

WindowsPath('c:/Users/SG199/Desktop/VS CODE REPOS/marketplace-safety-ml/data/raw/historical_data.csv')

In [3]:
df = pd.read_csv(DATA_PATH)

df.head()

,id,day,event_type,category,region,device,account_age_days,num_prev_listings,prev_reports_30d,verification_level,price,num_images,message_length,contains_off_platform,urgency_words,payment_attempt,time_to_first_response_min,is_suspicious
0,0,8,ad_post,other,urban,android,38.4,2,1,1,594.16,3,91,0,1,0,2.3,0
1,1,4,ad_post,fashion,urban,android,20.0,1,0,1,134.47,2,150,0,0,0,13.6,0
2,2,4,ad_post,other,metro,ios,46.7,3,1,1,198.52,3,72,0,0,0,4.2,0
3,3,3,ad_post,furniture,metro,android,44.3,3,1,2,141.20,3,0,0,0,0,19.8,0
4,4,1,ad_post,electronics,metro,web,211.2,5,0,0,81.39,3,9,0,0,1,23.3,0


In [4]:
print(f"Antal rader: {df.shape[0]}")
print(f"Antal kolumner: {df.shape[1]}")

df.info()

Antal rader: 12000
Antal kolumner: 18
<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          12000 non-null  int64  
 1   day                         12000 non-null  int64  
 2   event_type                  12000 non-null  str    
 3   category                    12000 non-null  str    
 4   region                      11660 non-null  str    
 5   device                      12000 non-null  str    
 6   account_age_days            12000 non-null  float64
 7   num_prev_listings           12000 non-null  int64  
 8   prev_reports_30d            12000 non-null  int64  
 9   verification_level          12000 non-null  int64  
 10  price                       11182 non-null  float64
 11  num_images                  12000 non-null  int64  
 12  message_length              12000 non-null  int64  
 13  cont

## 1. Dataförståelse

Datasetet innehåller 12 000 historiska händelser från en marketplace-plattform. Varje rad motsvarar en händelse, till exempel en annons eller ett meddelande.

Datan innehåller både numeriska signaler, som pris, kontots ålder och antal tidigare rapporter, samt kategoriska signaler, som region, device, kategori och event-typ.

Target-kolumnen heter `is_suspicious` och visar om händelsen tidigare har bedömts som misstänkt eller inte.